# Create flag parameter


In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`

# Create dimension table

In [0]:
df_Dim_Branch = spark.sql('''
SELECT DISTINCT(Branch_ID) AS Branch_ID, BranchName FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`
''')


Dim_model sink for initial and incremental load

In [0]:
# Check if the Gold table exists to decide how to load 'df_sink'
if spark.catalog.tableExists('sales_catalogue.gold.dim_branch'):
   
    df_sink = spark.sql('''
                SELECT Dim_Branch_key, Branch_ID, BranchName
                FROM sales_catalogue.gold.dim_branch
                ''')
else: 
    
    df_sink = spark.sql('''
                SELECT 1 as Dim_Branch_key, Branch_ID, BranchName
                FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`
                WHERE 1=0
                ''')

### filtering new and old records

In [0]:
df_filter = df_Dim_Branch.join(df_sink, df_Dim_Branch['Branch_ID'] == df_sink['Branch_ID'], 'left').select(df_sink['Dim_Branch_key'],df_Dim_Branch['Branch_ID'], df_Dim_Branch['BranchName'])

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Df_filter_old

In [0]:
df_filter_old = df_filter.filter(col('Dim_Branch_key').isNotNull())


df_filter_new

In [0]:
df_filter_new = df_filter.filter(col('Dim_Branch_key').isNull())


# Create surrogate key

fetch the max surrogate key from the column from the existing table

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("select max(Dim_Branch_key) from sales_catalogue.gold.dim_Branch")
    max_value = max_value_df.collect()[0][0]

# Create surrogate key column and add the max surrogate key

In [0]:
df_filter_new = df_filter_new.withColumn('Dim_Branch_key', max_value+monotonically_increasing_id())

In [0]:
df_filter_new.display()


Dim_Branch_key,Branch_ID,BranchName


FINAL JOIN NEW + OLD

In [0]:
df_final = df_filter_old.union(df_filter_new)

In [0]:
df_final.display()

SCD- 1 Upsert

In [0]:
from delta.tables import *

In [0]:
if spark.catalog.tableExists('sales_catalog.gold.dim_Branch'):
    delta_table = deltaTable.forPath("abfss://gold-adf@rgstorage.dfs.core.windows.net/car_sales_dim_Branch")
    delta_table.alias("trg").merge(df_final.alias("src"), "trg.Dim_Branch_key = src.Dim_Branch_key")\
                             .whenMatchedUpdateAll()\
                             .whenNotMatchedInsertAll()\
                             .execute()

else: 
    df_final.write.format('delta')\
        .mode('overwrite')\
            .option("path", "abfss://gold-adf@rgstorage.dfs.core.windows.net/car_sales_dim_Branch")\
            .saveAsTable('sales_catalogue.gold.dim_Branch')

In [0]:
%sql
select * from sales_catalogue.gold.dim_Branch